## Local nondegeneracy diagnostic for the TAVIE-SSG profile objective

This notebook checks a local curvature condition at the converged TAVIE-SSG fixed point $\xi^{\star}$. Recall that the TAVIE-SSG EM update has the fixed-point form:
$$
\xi^{(t+1)} = H(\xi^{(t)}), \quad H_{i}(\xi) = \sqrt{\kappa_{i}(\xi)}.
$$
At convergence, $\xi^{\star} = H(\xi^{\star})$ and $\kappa(\xi^{\star}) = \xi^{\star} \circ \xi^{\star}$. For both Type I and Type II SSG likelihoods, the gradient of the profile objective can be written as:
$$
\nabla_{\xi}\mathsf{L}(\xi) = D(\xi)\left\{\kappa(\xi) - \xi\circ \xi\right\},
$$
where $D(\xi) = \mathrm{diag}(d_1(\xi), \ldots, d_n(\xi))$ with:
$$
d_{i}(\xi) = \begin{cases}
\alpha A'(\xi_i), & \text{Type I SSG},\\
\alpha b_i A'(\xi_i), & \text{Type II SSG}.
\end{cases}
$$
Since $\kappa(\xi^{\star}) - \xi^{\star}\circ \xi^{\star} = 0$, differentiating the gradient at $\xi^{\star}$ gives:
$$
-\nabla^{2}_{\xi}\mathsf{L}(\xi) = D_{\star}\left\{2\Xi_{\star} - \nabla \kappa(\xi^{\star})\right\},
$$
where $D_{\star} = D(\xi^{\star})$ and $\Xi_{\star} = \mathrm{diag}(\xi_1^{\star}, \ldots, \xi_{n}^{\star})$. Because numerical differentiation may introduce slight asymmetry, we check the symmetric part:
$$
\lambda_{\min}\left(\mathrm{Sym}\left[D_{\star}\left\{2\Xi_{\star} - \nabla \kappa(\xi^{\star})\right\}\right]\right) > 0 \leftarrow \dagger\dagger,
$$
where $\mathrm{Sym}(M) = (M + M^{\top})/2$. If this smallest eigenvalue above is positive (the aforementioned condition enters as a new assumption, provided it gets verified for the candidate Type I and Type II SSG likelihoods), then the fitted TAVIE-SSG solution satisfies the local Hessian nondegeneracy condition:
$$
-\nabla^{2}_{\xi}\mathsf{L}(\xi^{\star}) \succ 0.
$$
Consequently, $f=-\mathsf{L}$ is locally strongly convex around $\xi^{\star}$, the profile objective satisfies a local PL/KL inequality with exponent $\Omega=1/2$, and Theorem 2 of the main manuscript yields local linear convergence of the TAVIE-SSG EM iterates.

We verify this particular condition in $\dagger\dagger$ for Student's-$t$ and Laplace Type I as well as the Negative-Binomial Type II SSG families.

In [1]:
# ============================================================
# verify local nondegeneracy condition
# for Student's-t, Laplace, and Negative-Binomial TAVIE-SSG
# ============================================================

import numpy as np
from numpy.linalg import inv, eigvalsh
from scipy.special import expit
import warnings

warnings.filterwarnings("ignore")
from TAVIE import *


# ============================================================
# Basic linear algebra helpers
# ============================================================

def symmetrize(M):
    return 0.5 * (M + M.T)


def stable_inv(M, jitter=1e-10, max_tries=6):
    """
    Numerically stable inverse for positive definite matrices.
    """
    M = np.asarray(M, dtype=float)
    I = np.eye(M.shape[0])
    for k in range(max_tries):
        try:
            return np.linalg.solve(M + (10**k) * jitter * I, I)
        except np.linalg.LinAlgError:
            continue
    return np.linalg.pinv(M)


def diag_XVX(X, V):
    """
    Computes diag(X V X^T) without forming the n x n matrix.
    """
    return np.einsum("ij,ij->i", X @ V, X)


# ============================================================
# A(xi) and A'(xi) for each family
# ============================================================

def A_laplace(xi):
    return -1.0 / (2.0 * xi)


def Aprime_laplace(xi):
    return 1.0 / (2.0 * xi**2)


def A_student(xi, nu):
    return -0.5 * (nu + 1.0) / (nu + xi**2)


def Aprime_student(xi, nu):
    return (nu + 1.0) * xi / (nu + xi**2)**2


def A_typeII_base(xi):
    """
    Base Type-II A(xi) = h'(xi^2) = - tanh(xi/2)/(4 xi).
    The TAVIE code uses b_i * A_base(xi_i).
    """
    return -np.tanh(xi / 2.0) / (4.0 * xi)


def Aprime_typeII_base(xi):
    """
    Derivative of A_base(xi) = - tanh(xi/2)/(4 xi).

    A_base'(xi)
    =
    [tanh(xi/2) - (xi/2) sech^2(xi/2)] / (4 xi^2).
    """
    t = np.tanh(xi / 2.0)
    sech2 = 1.0 / np.cosh(xi / 2.0)**2
    return (t - 0.5 * xi * sech2) / (4.0 * xi**2)


# ============================================================
# Kappa maps for Type I and Type II
# ============================================================

def kappa_typeI_from_xi(
    xi,
    X,
    y,
    A_func,
    V0,
    m0,
    a0,
    b0,
    alpha,
    **A_kwargs
):
    """
    Recomputes kappa(xi) for Type I SSG likelihoods:
        kappa_i(xi)
        =
        x_i^T V_alpha(xi) x_i
        +
        a_alpha / b_alpha(xi) * (y_i - x_i^T m_alpha(xi))^2.
    """
    xi = np.asarray(xi, dtype=float)
    n, p = X.shape

    V0_inv = stable_inv(V0)
    V0_inv_m0 = V0_inv @ m0
    m0_V0_inv_m0 = m0 @ V0_inv_m0
    a_alpha = a0 + n * alpha

    A_xi = A_func(xi, **A_kwargs)

    V_inv = V0_inv - 2.0 * alpha * X.T @ (X * A_xi[:, None])
    V = stable_inv(V_inv)

    m = V @ (V0_inv_m0 - 2.0 * alpha * (X.T * A_xi).dot(y))

    b_alpha = (
        b0
        - 2.0 * alpha * A_xi.dot(y**2)
        + m0_V0_inv_m0
        - m @ V_inv @ m
    )

    residual = y - X @ m
    kappa = diag_XVX(X, V) + (a_alpha / b_alpha) * residual**2

    return kappa


def kappa_typeII_from_xi(
    xi,
    X,
    avec,
    bvec,
    V0,
    m0,
    alpha
):
    """
    Recomputes kappa(xi) for Type II SSG likelihoods:
        kappa_i(xi)
        =
        x_i^T V_alpha(xi) x_i
        +
        (x_i^T m_alpha(xi))^2.
    """
    xi = np.asarray(xi, dtype=float)
    n, p = X.shape

    V0_inv = stable_inv(V0)
    V0_inv_m0 = V0_inv @ m0

    # Code convention: A_weight_i = b_i * A_base(xi_i).
    A_weight = bvec * A_typeII_base(xi)

    V_inv = V0_inv - 2.0 * alpha * X.T @ (X * A_weight[:, None])
    V = stable_inv(V_inv)

    var_inv_mean = V0_inv_m0 + alpha * X.T @ (avec - bvec / 2.0)
    m = V @ var_inv_mean

    eta_mean = X @ m
    kappa = diag_XVX(X, V) + eta_mean**2

    return kappa


# ============================================================
# Numerical Jacobian of kappa(xi)
# ============================================================

def finite_difference_jacobian_kappa(kappa_func, xi, rel_step=1e-5):
    """
    Central finite-difference Jacobian of kappa at xi.

    Since the Jacobian is n x n, keep n moderate, e.g. n <= 200.
    """
    xi = np.asarray(xi, dtype=float)
    n = xi.size

    J = np.zeros((n, n), dtype=float)

    for j in range(n):
        h = rel_step * (1.0 + abs(xi[j]))

        # Keep xi positive under central difference.
        if xi[j] - h <= 0:
            h = 0.45 * xi[j]

        xi_plus = xi.copy()
        xi_minus = xi.copy()

        xi_plus[j] += h
        xi_minus[j] -= h

        kp = kappa_func(xi_plus)
        km = kappa_func(xi_minus)

        J[:, j] = (kp - km) / (2.0 * h)

    return J


# ============================================================
# Main checker
# ============================================================

def check_local_nondegeneracy_typeI(
    fit_result,
    X_design,
    y_used,
    family,
    V0,
    m0,
    a0,
    b0,
    alpha,
    nu=None,
    rel_step=1e-5,
    verbose=True
):
    """
    Checks:
        lambda_min Sym[D_star {2 Xi_star - nabla kappa(xi_star)}] > 0

    for Type I SSG likelihoods: Laplace and Student's-t.
    """
    xi_star = np.asarray(fit_result["xi"], dtype=float)

    if family == "laplace":
        A_func = A_laplace
        Aprime = Aprime_laplace(xi_star)
        A_kwargs = {}
    elif family == "student":
        if nu is None:
            raise ValueError("nu must be supplied for Student's-t.")
        A_func = A_student
        Aprime = Aprime_student(xi_star, nu=nu)
        A_kwargs = {"nu": nu}
    else:
        raise ValueError("family must be 'laplace' or 'student'.")

    D_star = np.diag(alpha * Aprime)
    Xi_star = np.diag(xi_star)

    def kappa_func(z):
        return kappa_typeI_from_xi(
            z,
            X=X_design,
            y=y_used,
            A_func=A_func,
            V0=V0,
            m0=m0,
            a0=a0,
            b0=b0,
            alpha=alpha,
            **A_kwargs
        )

    J_kappa = finite_difference_jacobian_kappa(
        kappa_func,
        xi_star,
        rel_step=rel_step
    )

    M = symmetrize(D_star @ (2.0 * Xi_star - J_kappa))
    eigs = eigvalsh(M)
    lam_min = eigs[0]

    out = {
        "family": family,
        "lambda_min": lam_min,
        "condition_holds": bool(lam_min > 0),
        "eigenvalues": eigs,
        "matrix": M,
        "D_star": D_star,
        "Xi_star": Xi_star,
        "J_kappa": J_kappa,
    }

    if verbose:
        print("=" * 72)
        print(f"Type I family: {family}")
        print(f"n = {X_design.shape[0]}, p = {X_design.shape[1]}")
        print(f"min xi_star = {xi_star.min():.6e}")
        print(f"lambda_min = {lam_min:.10e}")
        print(f"condition holds? {lam_min > 0}")
        print("=" * 72)

    return out


def check_local_nondegeneracy_typeII(
    fit_result,
    X_design,
    y_used,
    r_used,
    family,
    V0,
    m0,
    alpha,
    rel_step=1e-5,
    verbose=True
):
    """
    Checks:
        lambda_min Sym[D_star {2 Xi_star - nabla kappa(xi_star)}] > 0

    for Type II SSG likelihoods. Here we use Negative-Binomial.
    """
    if family != "negbin":
        raise ValueError("This checker call is configured for family='negbin'.")

    xi_star = np.asarray(fit_result["xi"], dtype=float)

    # For Negative-Binomial in your class:
    # avec = r
    # bvec = r + y
    if np.isscalar(r_used):
        r_vec = np.full_like(y_used, float(r_used), dtype=float)
    else:
        r_vec = np.asarray(r_used, dtype=float)

    avec = r_vec
    bvec = r_vec + y_used

    D_star = np.diag(alpha * bvec * Aprime_typeII_base(xi_star))
    Xi_star = np.diag(xi_star)

    def kappa_func(z):
        return kappa_typeII_from_xi(
            z,
            X=X_design,
            avec=avec,
            bvec=bvec,
            V0=V0,
            m0=m0,
            alpha=alpha
        )

    J_kappa = finite_difference_jacobian_kappa(
        kappa_func,
        xi_star,
        rel_step=rel_step
    )

    M = symmetrize(D_star @ (2.0 * Xi_star - J_kappa))
    eigs = eigvalsh(M)
    lam_min = eigs[0]

    out = {
        "family": family,
        "lambda_min": lam_min,
        "condition_holds": bool(lam_min > 0),
        "eigenvalues": eigs,
        "matrix": M,
        "D_star": D_star,
        "Xi_star": Xi_star,
        "J_kappa": J_kappa,
    }

    if verbose:
        print("=" * 72)
        print(f"Type II family: {family}")
        print(f"n = {X_design.shape[0]}, p = {X_design.shape[1]}")
        print(f"min xi_star = {xi_star.min():.6e}")
        print(f"lambda_min = {lam_min:.10e}")
        print(f"condition holds? {lam_min > 0}")
        print("=" * 72)

    return out

In [2]:
# ============================================================
# Synthetic examples: Student's-t, Laplace, Negative-Binomial
# ============================================================

rng = np.random.default_rng(123)

# Keep n moderate because the condition uses an n x n Jacobian.
n = 2000
p = 8

X_raw = rng.normal(size=(n, p))
X_design = np.column_stack([np.ones(n), X_raw])
p_design = X_design.shape[1]

beta_true = rng.normal(scale=0.75, size=p_design)

alpha = 1.0
m0 = np.zeros(p_design)
V0 = np.eye(p_design)
a0 = 0.05
b0 = 0.05


# ============================================================
# 1. Student's-t Type I SSG
# ============================================================

nu = 5
tau2_true = 3.0
tau_true = np.sqrt(tau2_true)

eps_student = rng.standard_t(df=nu, size=n) / tau_true
y_student = X_design @ beta_true + eps_student

student_model = TAVIE_loc_scale(
    fit_intercept=False,
    scale_X=False,
    scale_y=False,
    family="student"
)

student_model.fit(
    X=X_design,
    y=y_student,
    prior_params=[m0, V0, a0, b0],
    alpha=alpha,
    maxiter=2000,
    tol=1e-10,
    verbose=True,
    nu=nu
)

student_check = check_local_nondegeneracy_typeI(
    fit_result=student_model.fitted_values,
    X_design=student_model.design_matrix,
    y_used=student_model.y,
    family="student",
    V0=V0,
    m0=m0,
    a0=a0,
    b0=b0,
    alpha=alpha,
    nu=nu,
    rel_step=1e-5,
    verbose=True
)


# ============================================================
# 2. Laplace Type I SSG
# ============================================================

tau2_true = 3.0
tau_true = np.sqrt(tau2_true)

# Laplace density in your paper: tau/2 exp{-tau |residual|}
# so residual scale is 1/tau.
eps_laplace = rng.laplace(loc=0.0, scale=1.0 / tau_true, size=n)
y_laplace = X_design @ beta_true + eps_laplace

laplace_model = TAVIE_loc_scale(
    fit_intercept=False,
    scale_X=False,
    scale_y=False,
    family="laplace"
)

laplace_model.fit(
    X=X_design,
    y=y_laplace,
    prior_params=[m0, V0, a0, b0],
    alpha=alpha,
    maxiter=2000,
    tol=1e-10,
    verbose=True
)

laplace_check = check_local_nondegeneracy_typeI(
    fit_result=laplace_model.fitted_values,
    X_design=laplace_model.design_matrix,
    y_used=laplace_model.y,
    family="laplace",
    V0=V0,
    m0=m0,
    a0=a0,
    b0=b0,
    alpha=alpha,
    rel_step=1e-5,
    verbose=True
)


# ============================================================
# 3. Negative-Binomial Type II SSG
# ============================================================

# Use a smaller coefficient scale to avoid extreme probabilities/counts.
beta_nb = rng.normal(scale=0.35, size=p_design)
eta = X_design @ beta_nb
prob_success = expit(eta)

r_nb = 5.0

# numpy negative_binomial(n, p) returns number of failures before n successes.
# This matches pmf proportional to p^r (1-p)^y.
y_nb = rng.negative_binomial(n=int(r_nb), p=prob_success, size=n).astype(float)

nb_model = TAVIE_type_II(
    fit_intercept=False,
    scale_X=False,
    family="negbin"
)

nb_model.fit(
    X=X_design,
    y=y_nb,
    r=r_nb,
    prior_params=[m0, V0],
    alpha=alpha,
    maxiter=2000,
    tol=1e-10,
    verbose=True
)

nb_check = check_local_nondegeneracy_typeII(
    fit_result=nb_model.fitted_values,
    X_design=nb_model.design_matrix,
    y_used=nb_model.y,
    r_used=nb_model.r,
    family="negbin",
    V0=V0,
    m0=m0,
    alpha=alpha,
    rel_step=1e-5,
    verbose=True
)

╭─ TAVIE Fit for student ─╮
│  Starting TAVIE fit!    │
╰─────────────────────────╯

Converged in 34 iterations.
Type I family: student
n = 2000, p = 9
min xi_star = 3.659173e-02
lambda_min = 6.4235401018e-04
condition holds? True


╭─ TAVIE Fit for laplace ─╮
│  Starting TAVIE fit!    │
╰─────────────────────────╯

Converged in 100 iterations.
Type I family: laplace
n = 2000, p = 9
min xi_star = 1.741570e-02
lambda_min = 1.3606057932e-01
condition holds? True


╭─ TAVIE Fit for negbin ─╮
│  Starting TAVIE fit!   │
╰────────────────────────╯

Converged in 29 iterations.
Type II family: negbin
n = 2000, p = 9
min xi_star = 2.264531e-02
lambda_min = 1.6169694174e-04
condition holds? True


In [3]:
# ============================================================
# Summary table
# ============================================================

import pandas as pd

summary = pd.DataFrame([
    {
        "family": "Student-t",
        "lambda_min": student_check["lambda_min"],
        "condition_holds": student_check["condition_holds"],
    },
    {
        "family": "Laplace",
        "lambda_min": laplace_check["lambda_min"],
        "condition_holds": laplace_check["condition_holds"],
    },
    {
        "family": "Negative-Binomial",
        "lambda_min": nb_check["lambda_min"],
        "condition_holds": nb_check["condition_holds"],
    },
])

print(summary)

for row in summary.itertuples(index=False):
    if row.condition_holds:
        print(
            f"{row.family}: condition verified. "
            f"lambda_min = {row.lambda_min:.6e} > 0. "
            "Therefore the local Hessian nondegeneracy condition holds numerically."
        )
    else:
        print(
            f"{row.family}: condition NOT verified. "
            f"lambda_min = {row.lambda_min:.6e}. "
            "This does not disprove convergence; it only means the stronger local "
            "Omega = 1/2 / local-PL diagnostic failed for this run."
        )

              family  lambda_min  condition_holds
0          Student-t    0.000642             True
1            Laplace    0.136061             True
2  Negative-Binomial    0.000162             True
Student-t: condition verified. lambda_min = 6.423540e-04 > 0. Therefore the local Hessian nondegeneracy condition holds numerically.
Laplace: condition verified. lambda_min = 1.360606e-01 > 0. Therefore the local Hessian nondegeneracy condition holds numerically.
Negative-Binomial: condition verified. lambda_min = 1.616969e-04 > 0. Therefore the local Hessian nondegeneracy condition holds numerically.


In [4]:
# ============================================================
# Robustness check over finite-difference step sizes
# ============================================================

steps = [1e-4, 3e-5, 1e-5, 3e-6]

robust_rows = []

for h in steps:
    out_student = check_local_nondegeneracy_typeI(
        fit_result=student_model.fitted_values,
        X_design=student_model.design_matrix,
        y_used=student_model.y,
        family="student",
        V0=V0,
        m0=m0,
        a0=a0,
        b0=b0,
        alpha=alpha,
        nu=nu,
        rel_step=h,
        verbose=False
    )

    out_laplace = check_local_nondegeneracy_typeI(
        fit_result=laplace_model.fitted_values,
        X_design=laplace_model.design_matrix,
        y_used=laplace_model.y,
        family="laplace",
        V0=V0,
        m0=m0,
        a0=a0,
        b0=b0,
        alpha=alpha,
        rel_step=h,
        verbose=False
    )

    out_nb = check_local_nondegeneracy_typeII(
        fit_result=nb_model.fitted_values,
        X_design=nb_model.design_matrix,
        y_used=nb_model.y,
        r_used=nb_model.r,
        family="negbin",
        V0=V0,
        m0=m0,
        alpha=alpha,
        rel_step=h,
        verbose=False
    )

    robust_rows.extend([
        {"family": "Student-t", "rel_step": h, "lambda_min": out_student["lambda_min"], "condition_holds": out_student["condition_holds"]},
        {"family": "Laplace", "rel_step": h, "lambda_min": out_laplace["lambda_min"], "condition_holds": out_laplace["condition_holds"]},
        {"family": "Negative-Binomial", "rel_step": h, "lambda_min": out_nb["lambda_min"], "condition_holds": out_nb["condition_holds"]},
    ])

robust_summary = pd.DataFrame(robust_rows)
robust_summary

,family,rel_step,lambda_min,condition_holds
0,Student-t,0.000100,0.000642,True
1,Laplace,0.000100,0.136061,True
2,Negative-Binomial,0.000100,0.000162,True
3,Student-t,0.000030,0.000642,True
4,Laplace,0.000030,0.136061,True
5,Negative-Binomial,0.000030,0.000162,True
6,Student-t,0.000010,0.000642,True
7,Laplace,0.000010,0.136061,True
8,Negative-Binomial,0.000010,0.000162,True
9,Student-t,0.000003,0.000642,True


Note that: $-\nabla^{2}_{\xi}\mathsf{L}(\xi) = D_{\star}\left\{2\Xi_{\star} - \nabla \kappa(\xi^{\star})\right\}$, we check the eigenspace (eigenvalues) of $D_{\star}$, $\Xi_{\star}$, and $\nabla \kappa(\xi^{\star})$ under Laplace, Student's-$t$, and Negative-Binomial SSG likelihoods.

In [6]:
# ============================================================
# Full working code:
# Eigenvalues of D_star, 2 Xi_star, and nabla kappa(xi_star)
# for Student-t, Laplace, and Negative-Binomial TAVIE-SSG
# ============================================================

import numpy as np
import pandas as pd
from numpy.linalg import eigvalsh, eigvals
from scipy.special import expit
import warnings

warnings.filterwarnings("ignore")

from TAVIE import *


# ============================================================
# Basic helpers
# ============================================================

def stable_inv(M, jitter=1e-10, max_tries=8):
    """
    Stable inverse/solve for positive definite matrices.
    """
    M = np.asarray(M, dtype=float)
    I = np.eye(M.shape[0])

    for k in range(max_tries):
        try:
            return np.linalg.solve(M + (10.0**k) * jitter * I, I)
        except np.linalg.LinAlgError:
            pass

    return np.linalg.pinv(M)


def diag_XVX(X, V):
    """
    Computes diag(X V X^T) without forming X V X^T.
    """
    return np.einsum("ij,ij->i", X @ V, X)


def sym(M):
    return 0.5 * (M + M.T)


# ============================================================
# A(xi) and A'(xi)
# ============================================================

def A_laplace(xi):
    return -1.0 / (2.0 * xi)


def Aprime_laplace(xi):
    return 1.0 / (2.0 * xi**2)


def A_student(xi, nu):
    return -0.5 * (nu + 1.0) / (nu + xi**2)


def Aprime_student(xi, nu):
    return (nu + 1.0) * xi / (nu + xi**2)**2


def A_typeII_base(xi):
    """
    Base Type-II A(xi):
        A(xi) = - tanh(xi / 2) / (4 xi).
    """
    return -np.tanh(xi / 2.0) / (4.0 * xi)


def Aprime_typeII_base(xi):
    """
    Derivative of A(xi) = - tanh(xi/2)/(4 xi).

    A'(xi)
    =
    [tanh(xi/2) - (xi/2) sech^2(xi/2)] / (4 xi^2).
    """
    t = np.tanh(xi / 2.0)
    sech2 = 1.0 / np.cosh(xi / 2.0)**2
    return (t - 0.5 * xi * sech2) / (4.0 * xi**2)


# ============================================================
# kappa maps
# ============================================================

def kappa_typeI_from_xi(
    xi,
    X,
    y,
    A_func,
    V0,
    m0,
    a0,
    b0,
    alpha,
    **A_kwargs
):
    """
    Type I:
        kappa_i(xi)
        =
        x_i^T V_alpha(xi) x_i
        +
        a_alpha / b_alpha(xi) * (y_i - x_i^T m_alpha(xi))^2.
    """
    xi = np.asarray(xi, dtype=float)
    n, p = X.shape

    V0_inv = stable_inv(V0)
    V0_inv_m0 = V0_inv @ m0
    m0_V0_inv_m0 = m0 @ V0_inv_m0
    a_alpha = a0 + n * alpha

    A_xi = A_func(xi, **A_kwargs)

    V_inv = V0_inv - 2.0 * alpha * X.T @ (X * A_xi[:, None])
    V = stable_inv(V_inv)

    m = V @ (V0_inv_m0 - 2.0 * alpha * (X.T * A_xi).dot(y))

    b_alpha = (
        b0
        - 2.0 * alpha * A_xi.dot(y**2)
        + m0_V0_inv_m0
        - m @ V_inv @ m
    )

    residual = y - X @ m
    kappa = diag_XVX(X, V) + (a_alpha / b_alpha) * residual**2

    return kappa


def kappa_typeII_from_xi(
    xi,
    X,
    avec,
    bvec,
    V0,
    m0,
    alpha
):
    """
    Type II:
        kappa_i(xi)
        =
        x_i^T V_alpha(xi) x_i
        +
        (x_i^T m_alpha(xi))^2.
    """
    xi = np.asarray(xi, dtype=float)

    V0_inv = stable_inv(V0)
    V0_inv_m0 = V0_inv @ m0

    A_weight = bvec * A_typeII_base(xi)

    V_inv = V0_inv - 2.0 * alpha * X.T @ (X * A_weight[:, None])
    V = stable_inv(V_inv)

    m = V @ (V0_inv_m0 + alpha * X.T @ (avec - bvec / 2.0))

    eta_mean = X @ m
    kappa = diag_XVX(X, V) + eta_mean**2

    return kappa


# ============================================================
# Finite-difference Jacobian of kappa
# ============================================================

def finite_difference_jacobian_kappa(kappa_func, xi, rel_step=1e-5):
    """
    Central finite-difference Jacobian of kappa at xi.

    Returns:
        J[i, j] = partial kappa_i / partial xi_j.
    """
    xi = np.asarray(xi, dtype=float)
    n = xi.size
    J = np.zeros((n, n), dtype=float)

    for j in range(n):
        h = rel_step * (1.0 + abs(xi[j]))

        # Keep xi_j - h positive.
        if xi[j] - h <= 0:
            h = 0.45 * xi[j]

        xi_plus = xi.copy()
        xi_minus = xi.copy()

        xi_plus[j] += h
        xi_minus[j] -= h

        kp = kappa_func(xi_plus)
        km = kappa_func(xi_minus)

        J[:, j] = (kp - km) / (2.0 * h)

    return J


# ============================================================
# Build D_star, 2Xi_star, and nabla kappa for Type I
# ============================================================

def build_three_matrices_typeI(
    fitted_values,
    X,
    y,
    family,
    V0,
    m0,
    a0,
    b0,
    alpha,
    nu=None,
    rel_step=1e-5
):
    """
    Returns D_star, 2Xi_star, and J_kappa = nabla kappa(xi_star)
    for Type I SSG likelihoods: Laplace and Student-t.
    """
    xi_star = np.asarray(fitted_values["xi"], dtype=float)

    if family == "laplace":
        A_func = A_laplace
        Aprime = Aprime_laplace(xi_star)
        A_kwargs = {}

    elif family == "student":
        if nu is None:
            raise ValueError("nu must be supplied for Student-t.")
        A_func = A_student
        Aprime = Aprime_student(xi_star, nu=nu)
        A_kwargs = {"nu": nu}

    else:
        raise ValueError("family must be 'laplace' or 'student'.")

    D_star = np.diag(alpha * Aprime)
    two_Xi_star = np.diag(2.0 * xi_star)

    def kappa_func(z):
        return kappa_typeI_from_xi(
            z,
            X=X,
            y=y,
            A_func=A_func,
            V0=V0,
            m0=m0,
            a0=a0,
            b0=b0,
            alpha=alpha,
            **A_kwargs
        )

    J_kappa = finite_difference_jacobian_kappa(
        kappa_func,
        xi_star,
        rel_step=rel_step
    )

    return D_star, two_Xi_star, J_kappa


# ============================================================
# Build D_star, 2Xi_star, and nabla kappa for Type II
# ============================================================

def build_three_matrices_typeII_negbin(
    fitted_values,
    X,
    y,
    r,
    V0,
    m0,
    alpha,
    rel_step=1e-5
):
    """
    Returns D_star, 2Xi_star, and J_kappa = nabla kappa(xi_star)
    for Negative-Binomial Type II SSG likelihood.
    """
    xi_star = np.asarray(fitted_values["xi"], dtype=float)

    if np.isscalar(r):
        r_vec = np.full_like(y, float(r), dtype=float)
    else:
        r_vec = np.asarray(r, dtype=float)

    avec = r_vec
    bvec = r_vec + y

    D_star = np.diag(alpha * bvec * Aprime_typeII_base(xi_star))
    two_Xi_star = np.diag(2.0 * xi_star)

    def kappa_func(z):
        return kappa_typeII_from_xi(
            z,
            X=X,
            avec=avec,
            bvec=bvec,
            V0=V0,
            m0=m0,
            alpha=alpha
        )

    J_kappa = finite_difference_jacobian_kappa(
        kappa_func,
        xi_star,
        rel_step=rel_step
    )

    return D_star, two_Xi_star, J_kappa


# ============================================================
# Eigenvalue extraction
# ============================================================

def eigenvalues_three_matrices(D_star, two_Xi_star, J_kappa):
    """
    Eigenvalues of:
        D_star,
        2Xi_star,
        nabla kappa(xi_star).

    D_star and 2Xi_star are symmetric diagonal, so eigvalsh is used.
    J_kappa is generally non-symmetric, so eigvals is used.
    """
    eig_D_star = eigvalsh(D_star)
    eig_2Xi_star = eigvalsh(two_Xi_star)
    eig_J_kappa = eigvals(J_kappa)

    return eig_D_star, eig_2Xi_star, eig_J_kappa


def compact_eigen_summary(family, eig_D, eig_2Xi, eig_J):
    return {
        "family": family,
        "min eig(D_star)": np.min(eig_D),
        "max eig(D_star)": np.max(eig_D),
        "min eig(2Xi_star)": np.min(eig_2Xi),
        "max eig(2Xi_star)": np.max(eig_2Xi),
        "min Re eig(nabla_kappa)": np.min(eig_J.real),
        "max Re eig(nabla_kappa)": np.max(eig_J.real),
        "max |Im eig(nabla_kappa)|": np.max(np.abs(eig_J.imag)),
        "spectral radius nabla_kappa": np.max(np.abs(eig_J)),
    }


# ============================================================
# Synthetic data and model fitting
# ============================================================

rng = np.random.default_rng(123)

# Keep n moderate because the Jacobian is n x n.
n = 2000
p = 8

X_raw = rng.normal(size=(n, p))
X_design = np.column_stack([np.ones(n), X_raw])
p_design = X_design.shape[1]

alpha = 1.0

m0 = np.zeros(p_design)
V0 = np.eye(p_design)
a0 = 0.05
b0 = 0.05

beta_true = rng.normal(scale=0.75, size=p_design)


# ============================================================
# 1. Student-t Type I SSG
# ============================================================

nu = 5
tau2_student = 3.0
tau_student = np.sqrt(tau2_student)

eps_student = rng.standard_t(df=nu, size=n) / tau_student
y_student = X_design @ beta_true + eps_student

student_model = TAVIE_loc_scale(
    fit_intercept=False,
    scale_X=False,
    scale_y=False,
    family="student"
)

student_model.fit(
    X=X_design,
    y=y_student,
    prior_params=[m0, V0, a0, b0],
    alpha=alpha,
    maxiter=2000,
    tol=1e-10,
    verbose=False,
    nu=nu
)

D_student, twoXi_student, J_student = build_three_matrices_typeI(
    fitted_values=student_model.fitted_values,
    X=student_model.design_matrix,
    y=student_model.y,
    family="student",
    V0=V0,
    m0=m0,
    a0=a0,
    b0=b0,
    alpha=alpha,
    nu=nu,
    rel_step=1e-5
)

eig_D_student, eig_2Xi_student, eig_J_student = eigenvalues_three_matrices(
    D_student,
    twoXi_student,
    J_student
)


# ============================================================
# 2. Laplace Type I SSG
# ============================================================

tau2_laplace = 3.0
tau_laplace = np.sqrt(tau2_laplace)

eps_laplace = rng.laplace(loc=0.0, scale=1.0 / tau_laplace, size=n)
y_laplace = X_design @ beta_true + eps_laplace

laplace_model = TAVIE_loc_scale(
    fit_intercept=False,
    scale_X=False,
    scale_y=False,
    family="laplace"
)

laplace_model.fit(
    X=X_design,
    y=y_laplace,
    prior_params=[m0, V0, a0, b0],
    alpha=alpha,
    maxiter=2000,
    tol=1e-10,
    verbose=False
)

D_laplace, twoXi_laplace, J_laplace = build_three_matrices_typeI(
    fitted_values=laplace_model.fitted_values,
    X=laplace_model.design_matrix,
    y=laplace_model.y,
    family="laplace",
    V0=V0,
    m0=m0,
    a0=a0,
    b0=b0,
    alpha=alpha,
    rel_step=1e-5
)

eig_D_laplace, eig_2Xi_laplace, eig_J_laplace = eigenvalues_three_matrices(
    D_laplace,
    twoXi_laplace,
    J_laplace
)


# ============================================================
# 3. Negative-Binomial Type II SSG
# ============================================================

beta_nb = rng.normal(scale=0.35, size=p_design)
eta_nb = X_design @ beta_nb
p_success = expit(eta_nb)

r_nb = 5.0

# numpy negative_binomial(n, p) gives failures before n successes.
# This matches NB pmf proportional to p^r (1-p)^y.
y_nb = rng.negative_binomial(
    n=int(r_nb),
    p=p_success,
    size=n
).astype(float)

nb_model = TAVIE_type_II(
    fit_intercept=False,
    scale_X=False,
    family="negbin"
)

nb_model.fit(
    X=X_design,
    y=y_nb,
    r=r_nb,
    prior_params=[m0, V0],
    alpha=alpha,
    maxiter=2000,
    tol=1e-10,
    verbose=False
)

D_nb, twoXi_nb, J_nb = build_three_matrices_typeII_negbin(
    fitted_values=nb_model.fitted_values,
    X=nb_model.design_matrix,
    y=nb_model.y,
    r=nb_model.r,
    V0=V0,
    m0=m0,
    alpha=alpha,
    rel_step=1e-5
)

eig_D_nb, eig_2Xi_nb, eig_J_nb = eigenvalues_three_matrices(
    D_nb,
    twoXi_nb,
    J_nb
)


# ============================================================
# Compact summary table
# ============================================================

summary = pd.DataFrame([
    compact_eigen_summary(
        "Student-t",
        eig_D_student,
        eig_2Xi_student,
        eig_J_student
    ),
    compact_eigen_summary(
        "Laplace",
        eig_D_laplace,
        eig_2Xi_laplace,
        eig_J_laplace
    ),
    compact_eigen_summary(
        "Negative-Binomial",
        eig_D_nb,
        eig_2Xi_nb,
        eig_J_nb
    ),
])

summary

,family,min eig(D_star),max eig(D_star),min eig(2Xi_star),max eig(2Xi_star),min Re eig(nabla_kappa),max Re eig(nabla_kappa),max |Im eig(nabla_kappa)|,spectral radius nabla_kappa
0,Student-t,0.008777,0.174284,0.073183,16.366490,-1.217838e-09,1.901647,1.208976e-09,1.901647
1,Laplace,0.009433,1648.496287,0.034831,14.560824,-1.295746e-09,1.998881,1.258494e-09,1.998881
2,Negative-Binomial,0.003179,2.953823,0.045291,6.207453,-5.037631e-10,1.627884,4.091142e-10,1.627884


In [7]:
# ============================================================
# Full eigenvalue arrays
# ============================================================

all_eigs = {
    "Student-t": {
        "D_star": eig_D_student,
        "2Xi_star": eig_2Xi_student,
        "nabla_kappa": eig_J_student,
    },
    "Laplace": {
        "D_star": eig_D_laplace,
        "2Xi_star": eig_2Xi_laplace,
        "nabla_kappa": eig_J_laplace,
    },
    "Negative-Binomial": {
        "D_star": eig_D_nb,
        "2Xi_star": eig_2Xi_nb,
        "nabla_kappa": eig_J_nb,
    },
}

for family, eigs in all_eigs.items():
    print("=" * 90)
    print(family)
    print("=" * 90)

    print("\nEigenvalues of D_star:")
    print(eigs["D_star"])

    print("\nEigenvalues of 2Xi_star:")
    print(eigs["2Xi_star"])

    print("\nEigenvalues of nabla kappa(xi_star):")
    print(eigs["nabla_kappa"])

    print("\n")

Student-t

Eigenvalues of D_star:
[0.00877731 0.00948043 0.00980077 ... 0.17428404 0.1742841  0.17428417]

Eigenvalues of 2Xi_star:
[ 0.07318347  0.09714965  0.09995176 ... 14.09394812 16.16767283
 16.36648997]

Eigenvalues of nabla kappa(xi_star):
[ 1.90164716e+00+0.00000000e+00j  9.00010069e-01+0.00000000e+00j
  8.76415575e-01+0.00000000e+00j ... -6.10504661e-12+2.76040966e-12j
 -6.10504661e-12-2.76040966e-12j -4.29823820e-12+0.00000000e+00j]


Laplace

Eigenvalues of D_star:
[9.43317878e-03 1.06259481e-02 1.20363053e-02 ... 6.47990190e+02
 8.24583866e+02 1.64849629e+03]

Eigenvalues of 2Xi_star:
[ 0.03483141  0.04924902  0.05555598 ... 12.89045955 13.71927469
 14.56082443]

Eigenvalues of nabla kappa(xi_star):
[ 1.99888054e+00+0.j  6.18820914e-01+0.j  5.65507477e-01+0.j ...
 -5.37010833e-12+0.j -2.54774640e-12+0.j -8.52084486e-12+0.j]


Negative-Binomial

Eigenvalues of D_star:
[0.0031788  0.00365376 0.00415519 ... 2.1323344  2.31090102 2.95382299]

Eigenvalues of 2Xi_star:
[0.04529